## Testing data Processing on 202407 data

In [1]:
import io
import os
from pyspark.sql import SparkSession
import glob

In [2]:
# Initialize local Spark session mimicking Databricks

spark = SparkSession.builder \
    .appName("CitiBike-CSV-Processing") \
    .getOrCreate()

In [3]:
# Read all extracted CSVs beginning with 2024 in the historical folder
bronze_historical_path = "../data/bronze/historical/"
# PySpark will automatically merge split CSVs from the same month
df_trips = spark.read.option("header", "true") \
    .csv(f"{bronze_historical_path}/2024*.csv")

# Profile the schema to map out data types
print("The new Schema:")
df_trips.printSchema()

The new Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)



In [4]:
df_trips.show(10)

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|6C3563ED9BD36F2B|electric_bike|2024-07-10 09:12:...|2024-07-10 09:18:...|   Front St & Jay St|         4895.03|Dock 72 Way & Mar...|       4804.02|         40.702461|        -73.986842|         40.69985|         -73.97141|       member|
|788C72113A42CACD| classic_bike|2024-07-12 07:35

In [5]:
import pyspark.sql.functions as F

print("--- EXPERIMENT A: SHAPE & NULL FORENSICS ---\n")

# 1. Check Row Count vs. Unique Primary Keys
total_rides = df_trips.count()
distinct_rides = df_trips.select("ride_id").distinct().count()
duplicate_ids = total_rides - distinct_rides

print(f"Total Captured Rides (July Sample): {total_rides:,}")
print(f"Distinct ride_ids:                  {distinct_rides:,}")
print(f"Duplicate Primary Keys:             {duplicate_ids:,}\n")

# 2. Perform a sweeping Null audit across all 13 strings
print("--- Explicit NULL Count per Column ---")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_trips.columns]
df_nulls = df_trips.select(*null_exprs)

# Transpose via Pandas purely for a clean, scannable vertical readout
pdf_null_audit = df_nulls.toPandas().T
pdf_null_audit.columns = ["null_count"]
pdf_null_audit["null_pct"] = (pdf_null_audit["null_count"] / total_rides) * 100

print(pdf_null_audit.round(3))

--- EXPERIMENT A: SHAPE & NULL FORENSICS ---

Total Captured Rides (July Sample): 4,722,896
Distinct ride_ids:                  4,722,896
Duplicate Primary Keys:             0

--- Explicit NULL Count per Column ---
                    null_count  null_pct
ride_id                      0     0.000
rideable_type                0     0.000
started_at                   0     0.000
ended_at                     0     0.000
start_station_name        2640     0.056
start_station_id          2640     0.056
end_station_name         12567     0.266
end_station_id           13575     0.287
start_lat                 2640     0.056
start_lng                 2640     0.056
end_lat                  13554     0.287
end_lng                  13554     0.287
member_casual                0     0.000


In [8]:
df_trips.select("started_at").orderBy("started_at").limit(100).toPandas().head(100)

,started_at
0,2024-06-30 00:52:38.502
1,2024-06-30 01:20:07.039
2,2024-06-30 01:27:01.691
3,2024-06-30 01:47:47.810
4,2024-06-30 02:22:21.774
...,...
95,2024-06-30 21:16:30.006
96,2024-06-30 21:16:46.143
97,2024-06-30 21:20:58.007
98,2024-06-30 21:28:23.085


In [9]:
print("--- EXPERIMENT B.2: THE FLEXIBLE ISO-8601 PATCH ---\n")

# 1. Cast using the native format-less ISO parser
df_temporal_patched = df_trips.select(
    "ride_id",
    "started_at",
    "ended_at"
).withColumn(
    "started_ts", F.to_timestamp("started_at")  # <-- Stripped the rigid format string!
).withColumn(
    "ended_ts", F.to_timestamp("ended_at")
).withColumn(
    "duration_seconds",
    F.unix_timestamp("ended_ts") - F.unix_timestamp("started_ts")
)

# 2. Check if the Golden Key unlatched the data
failed_timestamps_patched = df_temporal_patched.filter(
    F.col("started_ts").isNull() | F.col("ended_ts").isNull()
).count()

print(f"Catastrophic Parsing Failures (Post-Patch): {failed_timestamps_patched:,}\n")

# 3. Reveal the mathematical reality of July's ride durations
print("--- Trip Duration Summary (in Seconds) ---")
df_temporal_patched.select("duration_seconds").summary(
    "count", "min", "25%", "50%", "75%", "max"
).show()

--- EXPERIMENT B.2: THE FLEXIBLE ISO-8601 PATCH ---

Catastrophic Parsing Failures (Post-Patch): 0

--- Trip Duration Summary (in Seconds) ---
+-------+----------------+
|summary|duration_seconds|
+-------+----------------+
|  count|         4722896|
|    min|               5|
|    25%|             332|
|    50%|             580|
|    75%|            1016|
|    max|           90030|
+-------+----------------+



In [10]:
print("--- EXPERIMENT C: SPATIAL PRECISION & BORDER SANITY ---\n")

# 1. Cast coordinates to Double and round to standard 6 decimal places
df_spatial = df_trips.select(
    "ride_id",
    "start_station_name",
    "end_station_name",
    F.round(F.col("start_lat").cast("double"), 6).alias("start_lat"),
    F.round(F.col("start_lng").cast("double"), 6).alias("start_lng"),
    F.round(F.col("end_lat").cast("double"), 6).alias("end_lat"),
    F.round(F.col("end_lng").cast("double"), 6).alias("end_lng")
)

# 2. Audit coordinate distributions to verify WGS84 boundaries
print("--- WGS84 Coordinate Distribution Audit ---")
df_spatial.select("start_lat", "start_lng", "end_lat", "end_lng").summary(
    "count", "min", "50%", "max"
).show()

# 3. Check for rogue trips landing outside the approximate NYC/NJ Bounding Box
# Approximate NYC Metro Box: Lat 40.4 to 41.0, Lng -74.3 to -73.7
rogue_gps_trips = df_spatial.filter(
    (F.col("start_lat") < 40.4) | (F.col("start_lat") > 41.0) |
    (F.col("start_lng") < -74.3) | (F.col("start_lng") > -73.7)
).count()

print(f"Trips with rogue GPS coordinates outside NYC Metro box: {rogue_gps_trips:,}")

--- EXPERIMENT C: SPATIAL PRECISION & BORDER SANITY ---

--- WGS84 Coordinate Distribution Audit ---
+-------+---------+----------+---------+----------+
|summary|start_lat| start_lng|  end_lat|   end_lng|
+-------+---------+----------+---------+----------+
|  count|  4720256|   4720256|  4709342|   4709342|
|    min|40.633385|-74.071959|      0.0|  -74.0789|
|    50%| 40.73555|-73.978652|40.735367|-73.978652|
|    max|  40.8863| -73.84672|  40.8863|       0.0|
+-------+---------+----------+---------+----------+

Trips with rogue GPS coordinates outside NYC Metro box: 0


In [4]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

# Define your local data lake paths
bronze_historical_path = "../data/bronze/historical_tripdata"
silver_historical_path = "../data/silver/historical_tripdata"

def clean_bronze_trips_to_silver(df_raw):
    """
    Applies the 5-Point EDA Hygiene Protocol to raw Citi Bike trip logs.
    """
    return (
        df_raw
        # 1. ISO-8601 Temporal Casting
        .withColumn("started_at_ts", F.to_timestamp("started_at"))
        .withColumn("ended_at_ts", F.to_timestamp("ended_at"))
        .withColumn("trip_duration_seconds",
                    F.unix_timestamp("ended_at_ts") - F.unix_timestamp("started_at_ts"))

        # 2. Spatial Casting & 6-Decimal Precision
        .withColumn("start_lat", F.round(F.col("start_lat").cast(DoubleType()), 6))
        .withColumn("start_lng", F.round(F.col("start_lng").cast(DoubleType()), 6))
        .withColumn("end_lat", F.round(F.col("end_lat").cast(DoubleType()), 6))
        .withColumn("end_lng", F.round(F.col("end_lng").cast(DoubleType()), 6))

        # 3. Safe Imputation for Dockless & Valet Rides
        .withColumn("start_station_id", F.coalesce(F.col("start_station_id"), F.lit("UNASSIGNED")))
        .withColumn("start_station_name", F.coalesce(F.col("start_station_name"), F.lit("UNKNOWN")))
        .withColumn("end_station_id", F.coalesce(F.col("end_station_id"), F.lit("UNASSIGNED")))
        .withColumn("end_station_name", F.coalesce(F.col("end_station_name"), F.lit("UNKNOWN")))

        # 4. Temporal Helper Partitions for downstream logistics aggregation
        .withColumn("start_year", F.year("started_at_ts"))
        .withColumn("start_month", F.month("started_at_ts"))
        .withColumn("start_date", F.to_date("started_at_ts"))
        .withColumn("start_hour", F.hour("started_at_ts"))

        # 5. THE IRON GUARDRAILS (Applying our EDA discoveries)
        .filter(
            (F.col("trip_duration_seconds") >= 60) &         # Kill 5-sec phantom clicks
            (F.col("trip_duration_seconds") <= 86400) &      # Kill 25-hr force closes
            (F.col("start_lat").between(40.4, 41.0)) &       # NYC Geo-Fence (Start)
            (F.col("start_lng").between(-74.3, -73.7)) &
            (F.col("end_lat").between(40.4, 41.0)) &         # Kill Null Island (0.0, 0.0)
            (F.col("end_lng").between(-74.3, -73.7))
        )
        # Drop legacy raw strings to keep the Silver payload hyper-lean
        .drop("started_at", "ended_at")
    )

# --- EXECUTE THE PIPELINE ---
print("Reading Bronze CSVs...")
df_bronze = df_trips.limit(5000)

print("Applying Silver transformations...")
df_silver = clean_bronze_trips_to_silver(df_bronze)

# Audit the row-survival rate
bronze_count = df_bronze.count()
silver_count = df_silver.count()
dropped_rows = bronze_count - silver_count
print(f"Bronze Rows: {bronze_count:,} | Silver Rows: {silver_count:,} | Anomalies Purged: {dropped_rows:,}")

print("\nWriting to Silver Lakehouse (Partitioned Parquet)...")
df_silver.write \
    .mode("overwrite") \
    .partitionBy("start_year", "start_month") \
    .parquet(silver_historical_path)

print("SUCCESS: Silver Trip logs materialized.")

Reading Bronze CSVs...
Applying Silver transformations...
Bronze Rows: 5,000 | Silver Rows: 4,999 | Anomalies Purged: 1

Writing to Silver Lakehouse (Partitioned Parquet)...
SUCCESS: Silver Trip logs materialized.


In [5]:
df_silver = spark.read.parquet(silver_historical_path)

In [6]:
df_silver.limit(10).show()

+----------------+-------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+--------------------+--------------------+---------------------+----------+----------+----------+-----------+
|         ride_id|rideable_type|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|  end_lat|   end_lng|member_casual|       started_at_ts|         ended_at_ts|trip_duration_seconds|start_date|start_hour|start_year|start_month|
+----------------+-------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+--------------------+--------------------+---------------------+----------+----------+----------+-----------+
|6C3563ED9BD36F2B|electric_bike|   Front St & Jay St|         4895.03|Dock 72 Way & Mar...|       4804.02|40.702461|-73.986842| 40.69985| -73.97141|       member|2024-07-10 09:12:...|2